# Football Managerial Decision Analytics — Chapter 06
## Managerial patterns, evidence cards and conclusions

> **Data.** StatsBomb Open Data (https://github.com/hudl/open-data), provided by StatsBomb / Hudl, used for educational, non-commercial research. Raw data are downloaded from the official repository at run time and are not redistributed here. Every figure carries its chart ID from `docs/VISUALIZATION_CATALOGUE.md`.

This notebook is one chapter of the research record. All logic lives in the shared `src/` package; the cells call it. The same charts are available interactively in the Streamlit dashboard (`streamlit run app.py`).

In [ ]:
# Setup — locate (or fetch) the project code, import the shared package, load the workspace
import os, subprocess, sys, time
from pathlib import Path

REPO_URL = "https://github.com/blazingarrows1525/football-managerial-decision-analytics"
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "workspace.py").exists()), None)
if ROOT is None:  # opened on its own (e.g. Google Colab): fetch the project code once, next to this notebook
    ROOT = Path.cwd() / "football-managerial-decision-analytics"
    if not (ROOT / "src" / "workspace.py").exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")], check=True)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ.setdefault("FMA_PROJECT_ROOT", str(ROOT))  # data cache and outputs live inside the project folder

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from src.reporting.notebook_tools import (catalogue, explorer, implication, interpret, limitation, show, start,
                                          timed, timings)

pd.set_option("display.max_columns", 40, "display.width", 200, "display.float_format", "{:.3f}".format)
RUN_ADVANCED_GALLERY = True   # heavier, optional sections; the core story does not depend on them
SEED = 42                             # every resampling and model step takes this seed explicitly

ws = start(seed=SEED)                 # first run downloads and parses; later runs read the processed cache
B, T, FOCUS = ws.bundle, ws.tables, ws.focus
tm = ws.team_matches()                # the focus team's matches in date order
print(f"Advanced sections: {'on' if RUN_ADVANCED_GALLERY else 'off'} · Python {sys.version.split()[0]} · pandas {pd.__version__}")

## I · Managerial patterns

**Research question.** What do the recorded decisions show about selection, formation and substitution habits,
and how much of it is certain?

**Scope note.** This describes **observable decisions**. It does not measure a manager's ability or intentions
and cannot separate the manager from the squad he has. Every other manager in the data has two matches,
all against the focus team.

In [ ]:
prof = ws.manager_profiles()
mgr = ws.focus_manager()
print("Manager identity source:", B.manager_source)
display(prof[prof.manager == mgr].T)
show("I-186", ws)
show("I-187", ws)
r = prof[prof.manager == mgr].iloc[0]
interpret(f"{mgr} ({int(r.matches)} matches): {int(r.formations_used)} starting formations, {r.most_used_formation} in "
          f"{r.most_used_share_pct:.0f}% of matches; {r.mean_lineup_changes:.2f} changes to the XI between consecutive "
          f"analysed matches; {r.subs_per_match:.2f} substitutions per match with the first at a median of minute "
          f"{r.median_first_sub_minute:.0f}.")
limitation("Rotation is measured between league matches only; cup and European fixtures in between are missing.")

In [ ]:
show("I-192", ws)
show("I-193", ws)
show("I-195", ws)
small = prof[prof.matches < B.config.min_manager_matches]
interpret(f"{len(small)} of {len(prof)} managers have fewer than {B.config.min_manager_matches} matches. Their Wilson intervals "
          f"span most of the scale and shrinkage pulls them to the pooled value — they cannot be ranked or compared "
          f"with {mgr} as peers.")

In [ ]:
# Optional, heavier section — set RUN_ADVANCED_GALLERY = True in the setup cell to run it
if RUN_ADVANCED_GALLERY:
    mid = ws.default_match()
    show("I-198", ws, match_id=mid)
    show("I-190", ws)
    show("I-197", ws)
    show("G-163", ws)

## Evidence cards

Each card states the finding, the metric and value behind it, the scope, the supporting chart, why it may
matter and what the evidence cannot tell us. The text is assembled from computed values.

In [ ]:
from src.reporting import insight_cards
cards = insight_cards.build_cards(ws)
display(Markdown(insight_cards.cards_markdown(cards)))

## J · Answers to the research questions and hypotheses

In [ ]:
from src.analytics import descriptive as desc
from src.reporting.report_builder import results_snapshot
snap = results_snapshot(ws)
corr = ws.correlations().set_index("variable")
xg_side = corr.loc[["xg", "xg_against"], "spearman_rho"].abs().max()
answers = pd.DataFrame([
    ("H1 chance quality vs possession", f"|ρ| up to {xg_side:.2f} for xG for/against vs {abs(corr.loc['possession_proxy_pct', 'spearman_rho']):.2f} for possession proxy; "
     f"{snap['corr_significant_after_fdr']} of {len(corr)} variables significant after FDR", "direction supported; significance undecided at n = 34"),
    ("H2 results vs expectation", f"{snap['points']:.0f} points vs {snap['xpts']} expected points", "supported (model-based reference)"),
    ("H3 concentration of chance creation", f"top three = {snap['top3_chance_share']:.0%} of xG + xA", "supported"),
    ("H4 hub players", f"highest PageRank: {snap['hub_pagerank']}; highest betweenness: {snap['hub_betweenness']}", "supported (descriptive)"),
    ("H5 selection follows past selection", f"CV F1 {snap['cv_f1'][snap['best_model']]:.3f} vs {snap['cv_f1']['Baseline: started previous match']:.3f} baseline"
     if snap.get("model_fitted") else "model gate not met", "supported: small gain over the rule" if snap.get("model_fitted") else "not testable"),
    ("H6 substitution timing by score state", f"shortlist top-3 contains the actual substitute in {snap['sub_top3_share']:.0%} of cases", "descriptive only; confounded by team strength"),
], columns=["hypothesis", "evidence from this run", "verdict"])
display(answers)

## Conclusions, limitations and next steps

**What the evidence supports.** The focus team's results rest on a large and consistent advantage in chance
volume; the points total ran ahead of shot-based expectation; chance creation and build-up are concentrated in
a few players; selection is dominated by a stable core; substitutions are late and mostly made from in front.

**What it does not support.** Any causal claim; any statement about the league or other teams' seasons; any
ranking of managers; any conclusion about fitness, injuries or fatigue.

**Limitations.** One team-season of league matches; opponents seen twice; no injury, fitness, tracking,
age or market data; xG, expected points and Expected Threat are model outputs; heuristic scores are valid only
under their stated weights.

**Next steps.** (1) Add several complete league seasons for benchmarks independent of one club. (2) Add the
other competitions the team played to measure real rotation. (3) Obtain availability data from an authorised
source before modelling rotation. (4) Use tracking or 360 data for off-ball questions. (5) Re-test the venue and
correlation findings with more matches.

In [ ]:
# Machine-readable results of this run (the same numbers the reports are built from)
import json
print(json.dumps({k: v for k, v in snap.items() if k != "chart_status"}, indent=1, default=str, ensure_ascii=False))
display(timings().sort_values("seconds", ascending=False).head(10))